# Team 5 – Wanddickencheck am ursprünglichen IFC2X3-Modell

**Ziel:** Unsere Prüfregel A2 auf dem BIMcollab-Beispielmodell testen, nicht nur auf den IFC4-Lehrmodellen.

- Modell: `Team_5_BIMcollab_Example_STR.ifc` (**IFC2X3**)
- Auswahl: alle `IfcWall`-Objekte (einschliesslich `IfcWallStandardCase`)
- Datenpfad: `BaseQuantities.Width`
- Regel: **Wanddicke > 0 m**

**Unterschied zum bisherigen Notebook:** Das IFC2X3-Modell hat keinen `CH_Lehre.Scope = Holzrahmen-MVP`. Wir prüfen deshalb **alle Wände**. Das Modell enthält auch Wände aus anderen Materialien und ist **kein Nachweis für eine Holzrahmenbau-spezifische Prüfung**. Die ursprünglichen IFC4-Tests bleiben im bisherigen Notebook erhalten.

Das Notebook wurde für Team 5 als **Übertragbarkeitstest** erstellt. Google Colab benötigt Internetzugang.

## 1. IfcOpenShell installieren

In [ ]:
%pip -q install ifcopenshell==0.9.0.post1 pandas

import ifcopenshell
import ifcopenshell.util.element as element
import ifcopenshell.util.unit as unit
import pandas as pd
import math, hashlib, urllib.request
from pathlib import Path
from collections import Counter

print("IfcOpenShell:", ifcopenshell.version)
print("pandas:", pd.__version__)

## 2. Originalmodell herunterladen

Das Modell liegt bereits im **Team-5-Repository**. Der GitHub-Commit und die SHA-256-Prüfsumme sind festgelegt, damit alle dieselbe Datei prüfen.

In [ ]:
MODELLNAME = "Team_5_BIMcollab_Example_STR.ifc"
QUELLE = "https://raw.githubusercontent.com/beckr5/bxmid03-hs26-team-5/c5c18d5ba54b6ae3b077be6b69f73efaa2f8a711/models/" + MODELLNAME
ERWARTETE_SHA256 = "1d0d047a5519a2ea143b9d07625dcc94bd3a1c3b01cb86607b16d850f0a5081d"

datei = Path(MODELLNAME)
urllib.request.urlretrieve(QUELLE, datei)
pruefsumme = hashlib.sha256(datei.read_bytes()).hexdigest()
assert pruefsumme == ERWARTETE_SHA256, "IFC-Datei stimmt nicht mit dem dokumentierten Original überein!"
modell = ifcopenshell.open(str(datei))
assert modell.schema == "IFC2X3", f"Unerwartetes Schema: {modell.schema}"

print("Modell:", datei)
print("IFC-Schema:", modell.schema)
print("SHA-256:", pruefsumme)

## 3. Wanddicken prüfen

IfcOpenShell liest die Mengendaten **`BaseQuantities.Width`** aus. Die im Modell verwendeten Millimeter werden automatisch in Meter umgerechnet.

- `bestanden`: `Width > 0 m`
- `durchgefallen`: `Width <= 0 m` oder ungültiger Zahlenwert
- `fehlend`: kein `Width`-Wert vorhanden

Eine leere Auswahl zählt **nicht** als bestanden.

In [ ]:
REGEL_ID = "A2"
REGEL = "BaseQuantities.Width > 0 m"
DATENPFAD = "BaseQuantities.Width"

def bewerte(wert_m):
    if wert_m is None:
        return "fehlend", "Width fehlt"
    if not math.isfinite(wert_m):
        return "durchgefallen", "Width ist kein gültiger Zahlenwert"
    if wert_m > 0:
        return "bestanden", "Width ist grösser als 0 m"
    return "durchgefallen", "Width muss grösser als 0 m sein"

# Kleine Logiktests, unabhängig vom Modell
assert bewerte(None)[0] == "fehlend"
assert bewerte(0.0)[0] == "durchgefallen"
assert bewerte(0.1)[0] == "bestanden"

faktor = unit.calculate_unit_scale(modell)  # IFC-Längeneinheit -> Meter
waende = modell.by_type("IfcWall")
if not waende:
    raise RuntimeError("Keine IfcWall-Objekte gefunden. Modell und Auswahl prüfen.")

zeilen = []
for wand in waende:
    qtos = element.get_psets(wand, qtos_only=True)
    rohwert = qtos.get("BaseQuantities", {}).get("Width")
    try:
        width_m = float(rohwert) * faktor if rohwert is not None else None
    except (TypeError, ValueError, OverflowError):
        width_m = float("nan")
    status, grund = bewerte(width_m)
    zeilen.append({
        "GlobalId": wand.GlobalId,
        "Klasse": wand.is_a(),
        "Name": wand.Name or "",
        "Datenpfad": DATENPFAD,
        "Ist_Wert_m": round(width_m, 6) if width_m is not None and math.isfinite(width_m) else "",
        "Soll_Regel": REGEL,
        "Regel_ID": REGEL_ID,
        "Ergebnis": status,
        "Grund": grund,
        "Modell": MODELLNAME,
        "Modell_SHA256": pruefsumme,
        "IFC_Schema": modell.schema,
        "IfcOpenShell_Version": ifcopenshell.version,
    })

ergebnis = pd.DataFrame(zeilen)
assert ergebnis["GlobalId"].is_unique, "Doppelte GlobalIds in der Auswahl"
print(f"{len(waende)} Wände geprüft; Umrechnung in Meter: Faktor {faktor}")
print(ergebnis["Ergebnis"].value_counts().to_string())
display(ergebnis[["GlobalId", "Klasse", "Name", "Ist_Wert_m", "Ergebnis"]].head(15))

## 4. Ergebnis kontrollieren

Die folgenden Zahlen wurden **vorab durch eine direkte Analyse der IFC2X3-STEP-Struktur** ermittelt. Die Assertions vergleichen den tatsächlichen IfcOpenShell-Lauf mit diesem Befund. Sie sind **keine künstlich erzwungenen Ergebnisse**.

In [ ]:
# Unabhängiger Vorab-Befund für genau diese Modellversion:
assert len(ergebnis) == 114, "Anzahl Wände weicht vom Vorab-Befund ab"
assert (ergebnis["Ergebnis"] == "bestanden").sum() == 114, "Es gibt abweichende Prüfergebnisse"
assert (ergebnis["Ergebnis"] == "fehlend").sum() == 0, "Unerwartet fehlende Wanddicken"
assert (ergebnis["Ergebnis"] == "durchgefallen").sum() == 0, "Unerwartete ungültige Wanddicken"

verteilung = ergebnis["Ist_Wert_m"].value_counts().sort_index()
print("Wanddicken in Metern:")
print(verteilung.to_string())
assert (ergebnis["Ist_Wert_m"].round(4) == 0.1).sum() == 102
assert (ergebnis["Ist_Wert_m"].round(4) == 0.37).sum() == 12
print("Plausibilitätskontrolle: bestanden")

## 5. CSV exportieren

Der Bericht enthält die `GlobalId`, IFC-Klasse, Wanddicke, Prüfregel und das Ergebnis für jede Wand. Der CSV-Export entsteht **erst beim Ausführen** des Notebooks.

In [ ]:
ausgabe = Path("Team_5_A2_IFC2X3_original.csv")
ergebnis.to_csv(ausgabe, index=False, sep=";", encoding="utf-8-sig")
print("Gespeichert:", ausgabe, "| Zeilen:", len(ergebnis))
try:
    from google.colab import files
    files.download(str(ausgabe))
except ImportError:
    print("Lokale Ausführung: CSV liegt neben dem Notebook.")

### Einordnung für die Projektarbeit

Der Test zeigt die **Übertragbarkeit der A2-Prüfung auf ein IFC2X3-Modell** mit `BaseQuantities.Width`. Die 114 geprüften Wände haben im Originalmodell positive Dicken. Deshalb gibt es hier **keinen negativen Testfall im Modell**; diesen liefern weiterhin die beiden IFC4-Lehrmodelle im ersten Notebook. Die drei definierten Anforderungen A1, A2 und A3 sind noch nicht alle automatisiert umgesetzt.